# Rustren Studio — Transparent 3D Props & Device Mockups Baker (Colab GPU)

Pre-bakes **12 alpha-transparent 3D studio props & device mockups** (PNG RGBA, `768x768`) + `props_catalog.json` for [Rustren Studio](https://nvm.web.id/rustren/studio/).

- Zero HuggingFace token required (`stabilityai/sdxl-turbo` + `rembg`)
- Uses Colab's native Pillow + PNG RGBA (zero WebP plugin issues)
- Recommended runtime: **L4 / A100 / T4 GPU** (~1–2 minutes total)

In [ ]:
!pip -q install diffusers transformers accelerate safetensors rembg onnxruntime
import os, json, zipfile, torch
import numpy as np
from PIL import Image
from diffusers import AutoPipelineForText2Image
from rembg import remove, new_session

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'Device: {device} ({torch.cuda.get_device_name(0) if device == "cuda" else "CPU"})')

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=dtype,
    variant='fp16' if device == 'cuda' else None,
).to(device)
pipe.set_progress_bar_config(disable=True)

rembg_session = new_session('u2net')
print('Models ready!')

In [ ]:
OUT_DIR = 'rustren_3d_props'
MEDIA_DIR = os.path.join(OUT_DIR, 'media')
THUMB_DIR = os.path.join(OUT_DIR, 'thumbs')
os.makedirs(MEDIA_DIR, exist_ok=True)
os.makedirs(THUMB_DIR, exist_ok=True)

PROPS = [
    ('prop-smartphone', '3D Smartphone Mockup', ['device', 'phone', '3d'], 401,
     'minimalist sleek matte black titanium modern smartphone floating at slight 3d angle, clean dark glass screen, studio product render, isolated on solid pure white background, no shadow on floor, centered, octane render, no text'),
    ('prop-laptop', '3D Laptop Mockup', ['device', 'laptop', '3d'], 402,
     'minimalist sleek silver aluminum modern laptop open at 3d three-quarter angle, dark blank screen, studio product render, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-glass-card', 'Frosted Glass Card', ['ui', 'glass', '3d'], 403,
     'single minimalist frosted translucent glass credit card floating at subtle 3d perspective angle, clean metallic edge highlights, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-chrome-torus', 'Chrome 3D Torus', ['abstract', 'chrome', '3d'], 404,
     'single sculptural glossy liquid chrome silver 3d torus ring floating in air, studio reflections, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-glass-prism', 'Crystal Glass Prism', ['abstract', 'glass', '3d'], 405,
     'single minimalist clear crystal glass triangular prism floating in 3d space, subtle caustic edge refraction, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-matte-sphere', 'Matte Clay Sphere', ['abstract', 'clay', '3d'], 406,
     'single smooth minimalist matte dark charcoal 3d sphere with subtle soft rim light, architectural sculpture, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-gold-coin', '3D Metallic Coin', ['finance', 'badge', '3d'], 407,
     'single minimalist brushed gold metallic 3d coin token floating at slight angle, clean beveled rim, blank center, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-headphones', '3D Studio Headphones', ['audio', 'music', '3d'], 408,
     'minimalist matte black and brushed silver over-ear wireless studio headphones floating in 3d, luxury industrial design render, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-cube-glass', '3D Glass Cube', ['abstract', 'glass', '3d'], 409,
     'single minimalist rounded frosted glass 3d cube floating at isometric angle, clean studio specular highlights, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-podium', '3D Minimal Podium', ['stage', 'podium', '3d'], 410,
     'single minimalist neutral matte stone cylindrical 3d display podium pedestal, clean soft studio lighting, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-play-button', '3D Glass Play Icon', ['video', 'icon', '3d'], 411,
     'single minimalist glossy translucent orange-tinted glass 3d rounded play button triangle icon floating in air, studio lighting, isolated on solid pure white background, centered, octane render, no text'),
    ('prop-spark-star', '3D Chrome Star', ['ai', 'spark', '3d'], 412,
     'single minimalist glossy liquid chrome silver 4-point sparkle star 3d sculpture floating in air, clean studio reflections, isolated on solid pure white background, centered, octane render, no text'),
]

def center_on_square(rgba_img, size=768, pad_ratio=0.08):
    bbox = rgba_img.getbbox()
    if bbox:
        rgba_img = rgba_img.crop(bbox)
    max_inner = int(size * (1.0 - 2 * pad_ratio))
    w, h = rgba_img.size
    scale = min(max_inner / max(1, w), max_inner / max(1, h))
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = rgba_img.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    canvas.paste(resized, ((size - nw) // 2, (size - nh) // 2), resized)
    return canvas

catalog = []
print(f'Baking {len(PROPS)} transparent 3D props...')
for pid, title, tags, seed, prompt in PROPS:
    gen = torch.Generator(device=device).manual_seed(seed)
    raw = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, width=512, height=512, generator=gen).images[0]
    cut = remove(raw, session=rembg_session).convert('RGBA')
    full = center_on_square(cut, size=768)
    thumb = full.resize((256, 256), Image.Resampling.LANCZOS)

    media_rel = f'media/{pid}.png'
    thumb_rel = f'thumbs/{pid}.png'
    full.save(os.path.join(OUT_DIR, media_rel), format='PNG', optimize=True)
    thumb.save(os.path.join(OUT_DIR, thumb_rel), format='PNG', optimize=True)

    catalog.append({
        'id': pid,
        'kind': 'prop',
        'title': title,
        'tags': tags,
        'thumb': thumb_rel,
        'src': media_rel,
        'blend': 'normal',
        'tintable': False,
    })
    print(f'  [OK] {pid} ({title})')

with open(os.path.join(OUT_DIR, 'props_catalog.json'), 'w') as f:
    json.dump(catalog, f, indent=2)

zip_path = 'rustren_3d_props.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in sorted(files):
            full_p = os.path.join(root, fn)
            zf.write(full_p, os.path.relpath(full_p, OUT_DIR))

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'\nDONE! {len(catalog)} transparent 3D props baked into {zip_path} ({size_mb:.2f} MB)')

In [ ]:
from google.colab import files
files.download('rustren_3d_props.zip')